# import 

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np  


# load image

In [ ]:
input_file='coms4732_hw1_data/coms4732_hw1_data/cathedral.jpg'
output_file='coms4732_hw1_data/coms4732_hw1_data/aligned.jpg'

In [ ]:
img = np.asarray(Image.open(input_file)).astype(np.float32)

In [ ]:
print("shape:", img.shape, "dtype:", img.dtype, "min/max:", img.min(), img.max())

In [ ]:
channel_height = img.shape[0]//3
blue_channel = img[:channel_height,:]
green_channel = img[channel_height:channel_height*2,:]
red_channel = img[channel_height*2:channel_height*3,:]

# show image

In [ ]:
def show_image(red_channel, green_channel, blue_channel):
    if red_channel.dtype == np.uint16:
        base=65535
        plt.figure(figsize=(10,10))
    else:
        base=255
        plt.figure(figsize=(8,8))
    rgb = np.stack((red_channel/base, green_channel/base, blue_channel/base),axis=2)
    rgb=(rgb-rgb.min()) / (rgb.max()-rgb.min()+0.00001)
    plt.axis('off')
    plt.imshow(rgb)
    return rgb

# crop image

In [ ]:
def crop_in(channel, percent=10):
    border=channel.shape[1]//percent
    return channel[border:-border,border:-border]

In [ ]:
blue_channel_cropped = crop_in(blue_channel)
green_channel_cropped = crop_in(green_channel)
red_channel_cropped = crop_in(red_channel)

# L2

## L2 - single-scale alignment 

In [ ]:
def align_L2_single_scale(img_a, img_b, displacement=15):
    crop_img_a = crop_in(img_a)
    min_L2=float("inf")
    min_dx,min_dy=0,0
    for dx in range(-displacement,displacement+1):
        for dy in range(-displacement,displacement+1):
            roll_img_b=np.roll(img_b,(dx,dy),axis=(1,0))
            crop_img_b=crop_in(roll_img_b)
            L2=np.sqrt(np.sum((crop_img_a-crop_img_b)**2))
            if min_L2>L2:
                min_L2=L2
                min_dx=dx
                min_dy=dy
    return min_dx,min_dy
            

In [ ]:
green_dx,green_dy=align_L2_single_scale(blue_channel,green_channel)
red_dx,red_dy=align_L2_single_scale(blue_channel,red_channel)

green=np.roll(green_channel, (green_dx, green_dy),axis=(1,0))
red=np.roll(red_channel, (red_dx, red_dy),axis=(1,0))
print(f"green channel shift x:{green_dx} y:{green_dy}")
print(f"red channel shift x:{red_dx} y:{red_dy}")

rbg=show_image(red, green, blue_channel)

## L2 - multi-scale pyramid alignment 

In [ ]:
def downscale_mean(img,scale):
    (H,W)=img.shape
    H1=(H//scale)* scale
    W1=(W//scale)* scale
    img = img[:H1,:W1]
    img = img.reshape(H1//scale, scale, W1//scale,scale)
    return img.mean(axis=(1,3))

In [ ]:
def align_L2_pyramid_multi_scale(img_a, img_b, displacement=30):
    a=img_a
    b=img_b
    total_dx=0
    total_dy=0
    for i in range(1,5):
        scale=16//2**i
        d=displacement//i
        img_a=downscale_mean(a,scale)
        img_b=downscale_mean(b,scale)
        crop_img_a = crop_in(img_a)
        min_L2=float("inf")
        min_dx,min_dy=0,0
        for dx in range(-d,d+1):
            for dy in range(-d,d+1):
                roll_img_b=np.roll(img_b,(dx,dy),axis=(1,0))
                crop_img_b=crop_in(roll_img_b)
                L2=np.sqrt(np.sum((crop_img_a-crop_img_b)**2))
                if min_L2>L2:
                    min_L2=L2
                    min_dx=dx
                    min_dy=dy
        b=np.roll(b,(min_dx*scale,min_dy*scale),axis=(1,0))   
        total_dx+=min_dx*scale
        total_dy+=min_dy*scale
    return total_dx,total_dy

            


In [ ]:
green_dx,green_dy=align_L2_pyramid_multi_scale(blue_channel,green_channel)
red_dx,red_dy=align_L2_pyramid_multi_scale(blue_channel,red_channel)

green=np.roll(green_channel, (green_dx, green_dy),axis=(1,0))
red=np.roll(red_channel, (red_dx, red_dy),axis=(1,0))
print(f"green channel shift x:{green_dx} y:{green_dy}")
print(f"red channel shift x:{red_dx} y:{red_dy}")

rgb=show_image(red, green, blue_channel)

# NCC

## normalize

In [ ]:
def normalize(img):
    avg=img.mean()
    return (img-avg)/(np.sqrt(((img-avg)**2).sum())+0.00001)
    

## NCC - single-scale alignment 

In [ ]:
def align_NCC_single_scale(img_a, img_b, displacement=15):
    crop_img_a = crop_in(img_a)
    norm_a = normalize(crop_img_a)
    max_NCC=-float("inf")
    min_dx,min_dy=0,0
    for dx in range(-displacement,displacement+1):
        for dy in range(-displacement,displacement+1):
            roll_img_b=np.roll(img_b,(dx,dy),axis=(1,0))
            crop_img_b=crop_in(roll_img_b)
            norm_b=normalize(crop_img_b)
            NCC = (norm_a*norm_b).sum()
            if NCC>max_NCC:
                max_NCC=NCC
                min_dx=dx
                min_dy=dy
    return min_dx,min_dy
            


In [ ]:
green_dx,green_dy=align_NCC_single_scale(blue_channel,green_channel)
red_dx,red_dy=align_NCC_single_scale(blue_channel,red_channel)

green=np.roll(green_channel, (green_dx, green_dy),axis=(1,0))
red=np.roll(red_channel, (red_dx, red_dy),axis=(1,0))
print(f"green channel shift x:{green_dx} y:{green_dy}")
print(f"red channel shift x:{red_dx} y:{red_dy}")

rbg=show_image(red,green,blue_channel)

## NCC - multi-scale pyramid alignment 

In [ ]:
def align_NCC_pyramid_multi_scale(img_a, img_b, displacement=15):
    a=img_a
    b=img_b
    total_dx=0
    total_dy=0
    for i in range(1,5):
        scale=16//2**i
        d=displacement//i
        img_a=downscale_mean(a,scale)
        img_b=downscale_mean(b,scale)
        crop_img_a = crop_in(img_a)
        norm_a = normalize(crop_img_a)
        max_NCC=-float("inf")
        min_dx,min_dy=0,0
        for dx in range(-d,d+1):
            for dy in range(-d,d+1):
                roll_img_b=np.roll(img_b,(dx,dy),axis=(1,0))
                crop_img_b=crop_in(roll_img_b)
                norm_b=normalize(crop_img_b)
                NCC = (norm_a*norm_b).sum()
                if NCC>max_NCC:
                    max_NCC=NCC
                    min_dx=dx
                    min_dy=dy
        b=np.roll(b,(min_dx*scale,min_dy*scale),axis=(1,0))   
        total_dx+=min_dx*scale
        total_dy+=min_dy*scale
    return total_dx,total_dy
            


In [ ]:
green_dx,green_dy=align_NCC_pyramid_multi_scale(blue_channel,green_channel)
red_dx,red_dy=align_NCC_pyramid_multi_scale(blue_channel,red_channel)

green=np.roll(green_channel, (green_dx, green_dy),axis=(1,0))
red=np.roll(red_channel, (red_dx, red_dy),axis=(1,0))
print(f"green channel shift x:{green_dx} y:{green_dy}")
print(f"red channel shift x:{red_dx} y:{red_dy}")

rgb=show_image(red,green,blue_channel)

# save image

In [ ]:
rgb_uint8 = np.clip(rgb * 255, 0, 255).astype(np.uint8)

img_out = Image.fromarray(rgb_uint8)
img_out.save(output_file, quality=95)
